# S-CoT Model Inference on Google Colab
This notebook will authenticate with your GCP bucket, download the newest reasoning checkpoints from your TPU run, inject them into the base model via Qwix/Tunix, and allow you to test inference interactively.

**Make sure you select a T4 or L4 GPU Runtime in Colab!**

In [ ]:
# Install dependencies
!pip install -q 'google-tunix[prod]'
!pip install -q -U transformers
!pip install -q wandb huggingface_hub gcsfs datasets evaluate tqdm peft

In [ ]:
# Authenticate to access your GCS checkpoint bucket
from google.colab import auth
auth.authenticate_user()

# Download the checkpoints stored by your TPU watchdog
!mkdir -p /content/scot_checkpoints
!gsutil -m cp -r gs://tpu-builder1-scot-checkpoints/* /content/scot_checkpoints/

In [ ]:
import jax
import jax.numpy as jnp
from transformers import AutoTokenizer
from tunix.models.qwen2 import model as qwen_model
from tunix.models.qwen2 import params as qwen_params
from qwix._src.providers import lora as qwix_lora
from flax import nnx
from orbax import checkpoint as ocp
import os

# 1. Load Tokenizer
base_model_id = "Qwen/Qwen2.5-3B"
tokenizer = AutoTokenizer.from_pretrained(base_model_id)

# 2. Build Base Model
print("Building Base Model Architecture natively...")
config = qwen_model.ModelConfig.qwen2p5_3b()
mesh = jax.sharding.Mesh(jnp.array(jax.devices()).reshape((len(jax.devices()), 1)), ('fsdp', 'tp'))
base_weights_path = '' 
model = qwen_params.create_model_from_safe_tensors(base_weights_path, config, mesh, dtype=jnp.bfloat16, init_only=True)

# 3. Apply exact LoRA mapping
print("Wrapping model with LoRA...")
lora_provider = qwix_lora.LoraProvider(
    module_path=".*gate_proj|.*down_proj|.*up_proj",
    rank=16,
    alpha=32.0,
)
model_input = {
    'input_tokens': jnp.ones((1, 1024), dtype=jnp.int32),
    'positions': jnp.arange(1024, dtype=jnp.int32)[None, :],
    'cache': None
}
lora_model = qwix_lora.apply_lora_to_model(model, lora_provider, rngs=nnx.Rngs(0), **model_input)

# 4. Inject Restored Checkpoints from bucket!
ckpt_dir = "/content/scot_checkpoints"
print(f"Loading checkpoint states from {ckpt_dir}...")
checkpointer = ocp.StandardCheckpointer()
if os.path.exists(ckpt_dir) and any(os.scandir(ckpt_dir)):
    restored = checkpointer.restore(ckpt_dir)
    nnx.update(lora_model, restored)
    print("Checkpoint Successfully Loaded into Architecture!")
else:
    print("WARNING: Checkpoint directory was empty!")

In [ ]:
# Simple Inference Generation Function
def generate_inference(prompt_text, max_new_tokens=150):
    inputs = tokenizer(prompt_text, return_tensors="jax")
    
    input_ids = inputs["input_ids"]
    attention_mask = inputs["attention_mask"]
    
    current_ids = input_ids
    print(f"Q: {prompt_text}\nA:")
    
    for _ in range(max_new_tokens):
        positions = jnp.arange(current_ids.shape[-1])[None, :]
        
        logits = lora_model(input_tokens=current_ids, input_mask=attention_mask, positions=positions)
        next_token_logits = logits[0, -1, :]
        next_token = jnp.argmax(next_token_logits)
        
        if next_token == tokenizer.eos_token_id:
            break
            
        current_ids = jnp.concatenate([current_ids, jnp.array([[next_token]])], axis=1)
        attention_mask = jnp.concatenate([attention_mask, jnp.array([[1]])], axis=1)
        
        # Stream decode
        import sys
        char = tokenizer.decode([next_token])
        sys.stdout.write(char)
        sys.stdout.flush()
    print()

questions = [
    "What is the mathematical structure behind String Theory?",
    "If a sequence is defined by a_n = 2 * a_{n-1} + 3, and a_1 = 1, find a_4. Think step by step.",
    "Explain the concept of attention mechanisms in Transformers natively.",
    "Solve the linear equation 5x + 12 = 32.",
    "How does the S-CoT reasoning framework differ from flat distillation?"
]

for q in questions:
    generate_inference(q)
    print("\n" + "="*50 + "\n")